# FusionCore v0 — Phase 5d: Official Test Set Inference — NHITS

**Notebook:** `05d_NHITS_Inference.ipynb`  
**Phase:** 5 of 5 (Part D)  
**Objective:** NHITS inference on Official Test Set using NeuralForecast.

**Note:** PatchTST (Nie et al. 2023) was originally specified in CLAUDE.md but does not
support exogenous variables in NeuralForecast. NHITS (Challu et al. 2023) replaces it
as the MLP-based SOTA model with `hist_exog_list` support.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1 — Environment Setup (Run First)
# ══════════════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2 — Dependency Installation
# ══════════════════════════════════════════════════════════════════════════════

%%capture
!pip install neuralforecast

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3 — Imports, Constants & Palette
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import os
import gc

DRIVE_ROOT     = Path('/content/drive/MyDrive/PI')
OUTPUTS_DIR    = DRIVE_ROOT / 'FusionCore' / 'v0' / 'outputs'
CHECKPOINT_DIR = DRIVE_ROOT / 'FusionCore' / 'v0' / 'checkpoints'

RUL_CAP      = 125
RANDOM_STATE = 42
UNIT_KEY     = ['subset_origin', 'unit_id']
CMAPSS_SUBSETS = ['FD001', 'FD002', 'FD003', 'FD004']

np.random.seed(RANDOM_STATE)

FC_DARK_BLUE = '#0D1B2A'; FC_NAVY = '#1B3A5C'; FC_ORANGE = '#D96A1B'
FC_DEEP_RED = '#9B1B30'; FC_STEEL = '#4A6274'; FC_CHARCOAL = '#2D2D2D'

plt.rcParams.update({
    'figure.figsize': (14, 5), 'figure.dpi': 150, 'savefig.dpi': 300,
    'savefig.bbox': 'tight', 'axes.spines.top': False, 'axes.spines.right': False,
})

from sklearn.metrics import mean_squared_error, mean_absolute_error

def compute_nasa_score(y_true, y_pred):
    d = y_pred - y_true
    return float(np.sum(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)))

import torch
torch.manual_seed(RANDOM_STATE)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4 — Load Test Set Artefacts from 05a
# ══════════════════════════════════════════════════════════════════════════════

X_test    = pd.read_parquet(OUTPUTS_DIR / 'X_test.parquet')
X_test_nn = pd.read_parquet(OUTPUTS_DIR / 'X_test_nn.parquet')
meta_test = pd.read_parquet(OUTPUTS_DIR / 'meta_test.parquet')
y_test_df = pd.read_parquet(OUTPUTS_DIR / 'y_test.parquet')

feature_manifest = pd.read_csv(OUTPUTS_DIR / 'feature_manifest.csv')
feature_names_91 = list(feature_manifest['feature'])
nn_feature_names = joblib.load(OUTPUTS_DIR / 'nn_feature_names.pkl')

# Build last-cycle index and engine metadata.
test_last_idx = meta_test.groupby(UNIT_KEY)['cycle'].idxmax()
test_last = meta_test.loc[test_last_idx].reset_index(drop=True)
test_engine_meta = test_last[['subset_origin', 'unit_id']].merge(
    y_test_df, on=['subset_origin', 'unit_id'], how='left'
)
y_test = test_engine_meta['RUL'].values

print(f'X_test: {X_test.shape}, X_test_nn: {X_test_nn.shape}')
print(f'Test engines: {len(y_test)}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5 — Load NHITS Checkpoint & Build Test DataFrame
# ══════════════════════════════════════════════════════════════════════════════

from neuralforecast import NeuralForecast

nf_loaded = NeuralForecast.load(str(CHECKPOINT_DIR / 'nhits_best'))
print('✔ NHITS checkpoint loaded.')

# Override accelerator to match current runtime (checkpoint may have been
# trained on GPU but inference can run on CPU or vice versa).
device = 'gpu' if torch.cuda.is_available() else 'cpu'
for model in nf_loaded.models:
    model.trainer_kwargs['accelerator'] = device
print(f'  Inference device: {device}')

rul_lookup = y_test_df.set_index(['subset_origin', 'unit_id'])['RUL'].to_dict()

test_nf = pd.concat([
    meta_test.reset_index(drop=True),
    X_test_nn.reset_index(drop=True),
], axis=1)
test_nf['unique_id'] = (
    test_nf['subset_origin'].astype(str) + '_' +
    test_nf['unit_id'].astype(str)
)
test_nf = test_nf.rename(columns={'cycle': 'ds'})
test_nf = test_nf.sort_values(['unique_id', 'ds']).reset_index(drop=True)

# Reconstruct RUL column.
nf_rul = []
for eid, grp in test_nf.groupby('unique_id'):
    parts = eid.split('_', 1)
    so, uid = parts[0], int(parts[1])
    rul_last = rul_lookup.get((so, uid), RUL_CAP)
    n = len(grp)
    rul_series = np.clip(
        rul_last + np.arange(n - 1, -1, -1).astype(np.float32), 0, RUL_CAP
    )
    nf_rul.extend(rul_series.tolist())
test_nf['y'] = nf_rul

# Keep only required columns.
nf_cols = ['unique_id', 'ds', 'y'] + nn_feature_names
test_nf = test_nf[nf_cols]
print(f'Test NF: {test_nf.shape} ({test_nf["unique_id"].nunique()} engines)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6 — NHITS Inference
# ══════════════════════════════════════════════════════════════════════════════

preds = nf_loaded.predict(df=test_nf).reset_index()
pred_col = [c for c in preds.columns if c not in ['unique_id', 'ds']][0]

# Physical clipping — RUL cannot be negative or exceed the cap.
preds[pred_col] = preds[pred_col].clip(lower=0, upper=RUL_CAP)

# Last-cycle prediction per engine.
last_preds = preds.sort_values(['unique_id', 'ds']).groupby('unique_id').last()
last_preds[['subset_origin', 'uid_str']] = (
    pd.DataFrame(last_preds.index.str.split('_', n=1).tolist(),
                 index=last_preds.index, columns=['subset_origin', 'uid_str'])
)
last_preds['unit_id'] = last_preds['uid_str'].astype(int)

merged = test_engine_meta[['subset_origin', 'unit_id']].merge(
    last_preds[['subset_origin', 'unit_id', pred_col]].reset_index(drop=True),
    on=['subset_origin', 'unit_id'], how='left',
)
y_pred_nhits = merged[pred_col].values

rmse = float(np.sqrt(mean_squared_error(y_test, y_pred_nhits)))
mae  = float(mean_absolute_error(y_test, y_pred_nhits))
nasa = compute_nasa_score(y_test, y_pred_nhits)

print(f'NHITS Official Test Set Results')
print(f'  RMSE:       {rmse:.4f}')
print(f'  MAE:        {mae:.4f}')
print(f'  NASA Score: {nasa:,.1f}')
print(f'\n  Raw prediction range: [{preds[pred_col].min():.2f}, {preds[pred_col].max():.2f}]')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7 — Save 05d Outputs
# ══════════════════════════════════════════════════════════════════════════════

joblib.dump({
    'y_pred_nhits': y_pred_nhits,
    'rmse': rmse, 'mae': mae, 'nasa_score': nasa,
}, OUTPUTS_DIR / 'phase5d_nhits_predictions.pkl')

print('Phase 5d outputs persisted.')
print(f'✔ Notebook 05d complete.')